<a href="https://colab.research.google.com/github/Panperception/UG_2025_QRC/blob/main/lnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Liquid Neural Network (LNN) on MNIST

This notebook demonstrates a simple implementation of a **Liquid Neural Network (LNN)** using PyTorch.

Key ideas:

- Continuous-time neural dynamics
- Adaptive neuron time constants
- Sequential processing of images

The MNIST dataset is interpreted as a **sequence of 28 time steps**, where each row of the image is processed sequentially.

We implement a simplified version of the **Liquid Time-Constant Network (LTC)** architecture.

In [ ]:
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

## Device Configuration

We use GPU if available for faster training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Load MNIST Dataset

Each MNIST image is **28 × 28 pixels**.

We treat the image as a **sequence of 28 time steps**, where each step processes one row of pixels.

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

## Liquid Neural Layer

The liquid neuron follows continuous-time dynamics:

dx/dt = -x / τ(x,u) + f(Wx + Uu)

where

- x = hidden state
- τ(x,u) = adaptive time constant
- W = recurrent weight matrix
- U = input weight matrix

We discretize this differential equation using **Euler integration**.

In [ ]:
class LiquidLayer(nn.Module):

    def __init__(self, input_size, hidden_size):
        super(LiquidLayer, self).__init__()

        self.hidden_size = hidden_size

        self.W = nn.Linear(hidden_size, hidden_size)
        self.U = nn.Linear(input_size, hidden_size)

        self.W_tau = nn.Linear(hidden_size, hidden_size)
        self.U_tau = nn.Linear(input_size, hidden_size)

        self.activation = torch.tanh

    def forward(self, x_seq):

        batch_size, seq_len, _ = x_seq.shape
        h = torch.zeros(batch_size, self.hidden_size).to(x_seq.device)

        for t in range(seq_len):

            u = x_seq[:, t, :]

            tau = torch.sigmoid(self.W_tau(h) + self.U_tau(u)) + 0.1

            dh = -h / tau + self.activation(self.W(h) + self.U(u))

            h = h + dh * 0.1

        return h

## Liquid Neural Network Model

The architecture:

Image → Sequence → Liquid Layer → Fully Connected Layer → Digit Prediction

In [ ]:
class LNN(nn.Module):

    def __init__(self, input_size=28, hidden_size=64, num_classes=10):
        super(LNN, self).__init__()

        self.liquid = LiquidLayer(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        x = x.squeeze(1)  # (batch, 28, 28)

        h = self.liquid(x)

        out = self.fc(h)

        return out

## Initialize Model

In [ ]:
model = LNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Training the Liquid Neural Network

In [ ]:
epochs = 5

for epoch in range(epochs):

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

## Model Evaluation

In [ ]:
test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)